In [3]:
# =============================================================================
# LAB 09 - Perceptron / MLP tuning, Classifier Comparison, Pipeline,
#          LIME, SHAP, Grid-vs-Random Search comparison
# Dataset: marketing_campaign sheet of "Lab Session Data (1) (1).xlsx"
# Target : Response (binary classification)
# =============================================================================

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (train_test_split, RandomizedSearchCV,
                                     GridSearchCV)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
from sklearn.exceptions import ConvergenceWarning
from scipy.stats import randint, loguniform

# --- silence benign third-party / deprecation warnings -----------------------
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*IProgress not found.*")
warnings.filterwarnings("ignore", message=".*select_dtypes.*")
warnings.filterwarnings("ignore", message=".*probability.*deprecated.*")
warnings.filterwarnings("ignore", message=".*max_samples.*")

# --- optional libs -----------------------------------------------------------
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except Exception:
    HAS_CAT = False

try:
    import lime
    import lime.lime_tabular as lt
    HAS_LIME = True
except Exception:
    HAS_LIME = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False


# -----------------------------------------------------------------------------
# 1. DATA LOADING & CLEANING  (FIX: uniform dtype for Dt_Customer)
# -----------------------------------------------------------------------------
def load_data(path, sheet_name="marketing_campaign"):
    """Load and clean the marketing campaign sheet."""
    df = pd.read_excel(path, sheet_name=sheet_name)
    df.columns = [c.strip() for c in df.columns]

    # --- Fix mixed datetime/str in Dt_Customer ---------------------------
    if "Dt_Customer" in df.columns:
        dt = pd.to_datetime(df["Dt_Customer"], errors="coerce", dayfirst=True)
        df["Cust_Year"]  = dt.dt.year
        df["Cust_Month"] = dt.dt.month
        df["Cust_Day"]   = dt.dt.day
        # Explicit missingness flag so imputer doesn't hide NaT rows
        df["Cust_Date_Missing"] = dt.isna().astype(int)
        df = df.drop(columns=["Dt_Customer"])
    # ---------------------------------------------------------------------

    # Drop ID-like / constant columns
    drop_cols = [c for c in ["ID", "Z_CostContact", "Z_Revenue"] if c in df.columns]
    df = df.drop(columns=drop_cols)

    df["Income"]   = pd.to_numeric(df["Income"], errors="coerce")
    df["Response"] = df["Response"].astype(int)

    # Defensive: any remaining text column -> uniform str dtype
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype(str)

    return df


def split_features_target(df, target="Response"):
    """Separate X and y."""
    X = df.drop(columns=[target])
    y = df[target]
    return X, y


def build_preprocessor(X):
    """ColumnTransformer for numeric + categorical features."""
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot",  OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])
    return preprocessor, num_cols, cat_cols


# -----------------------------------------------------------------------------
# 2. A2 - Tune Perceptron & MLP via RandomizedSearchCV
# -----------------------------------------------------------------------------
def tune_perceptron(preprocessor, X_train, y_train, n_iter=15, cv=5, seed=42):
    """RandomizedSearchCV for Perceptron."""
    pipe = Pipeline([
        ("prep", preprocessor),
        ("clf",  Perceptron(random_state=seed)),
    ])
    param_dist = {
        "clf__penalty":  [None, "l2", "l1", "elasticnet"],
        "clf__alpha":    loguniform(1e-5, 1e-1),
        "clf__eta0":     loguniform(1e-4, 1.0),
        "clf__max_iter": randint(200, 2000),
        "clf__tol":      loguniform(1e-5, 1e-2),
    }
    rs = RandomizedSearchCV(pipe, param_dist, n_iter=n_iter, cv=cv,
                            scoring="f1", random_state=seed, n_jobs=-1,
                            error_score="raise")
    rs.fit(X_train, y_train)
    return rs


def tune_mlp(preprocessor, X_train, y_train, n_iter=12, cv=5, seed=42):
    """RandomizedSearchCV for MLPClassifier."""
    pipe = Pipeline([
        ("prep", preprocessor),
        ("clf",  MLPClassifier(random_state=seed, max_iter=400)),
    ])
    param_dist = {
        "clf__hidden_layer_sizes":  [(32,), (64,), (32, 16), (64, 32), (64, 32, 16)],
        "clf__activation":          ["relu", "tanh"],
        "clf__solver":              ["adam", "sgd"],
        "clf__alpha":               loguniform(1e-5, 1e-1),
        "clf__learning_rate_init":  loguniform(1e-4, 1e-1),
    }
    rs = RandomizedSearchCV(pipe, param_dist, n_iter=n_iter, cv=cv,
                            scoring="f1", random_state=seed, n_jobs=-1,
                            error_score="raise")
    rs.fit(X_train, y_train)
    return rs


# -----------------------------------------------------------------------------
# 3. A3 - Classifier comparison
# -----------------------------------------------------------------------------
def evaluate_model(model, X_test, y_test):
    """Compute classification metrics for a fitted model."""
    y_pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)
    return {
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall":    recall_score(y_test, y_pred, zero_division=0),
        "F1":        f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC":   roc_auc_score(y_test, y_score),
        "Confusion": confusion_matrix(y_test, y_pred),
    }


def build_classifier_pipelines(preprocessor, seed=42):
    """One Pipeline per classifier, pre-wired with the same preprocessor."""
    models = {
        "Perceptron":   Pipeline([("prep", preprocessor),
                                  ("clf", Perceptron(max_iter=1000, random_state=seed))]),
        "MLP":          Pipeline([("prep", preprocessor),
                                  ("clf", MLPClassifier(hidden_layer_sizes=(64, 32),
                                                        max_iter=500,
                                                        random_state=seed))]),
        # CalibratedClassifierCV wraps SVC to expose predict_proba (avoids the
        # `probability=True` deprecation warning in sklearn >= 1.9)
        "SVM":          Pipeline([("prep", preprocessor),
                                  ("clf", CalibratedClassifierCV(
                                              SVC(random_state=seed),
                                              ensemble=False, cv=3))]),
        "DecisionTree": Pipeline([("prep", preprocessor),
                                  ("clf", DecisionTreeClassifier(random_state=seed))]),
        "RandomForest": Pipeline([("prep", preprocessor),
                                  ("clf", RandomForestClassifier(n_estimators=200,
                                                                 random_state=seed,
                                                                 n_jobs=-1))]),
        "AdaBoost":     Pipeline([("prep", preprocessor),
                                  ("clf", AdaBoostClassifier(random_state=seed))]),
        "NaiveBayes":   Pipeline([("prep", preprocessor), ("clf", GaussianNB())]),
    }
    if HAS_XGB:
        models["XGBoost"] = Pipeline([("prep", preprocessor),
                                      ("clf", XGBClassifier(eval_metric="logloss",
                                                            random_state=seed,
                                                            n_jobs=-1))])
    if HAS_CAT:
        models["CatBoost"] = Pipeline([("prep", preprocessor),
                                       ("clf", CatBoostClassifier(verbose=0,
                                                                  random_state=seed))])
    return models


def compare_classifiers(models, X_train, y_train, X_test, y_test):
    """Fit every model, collect metrics into a DataFrame."""
    rows, fitted = [], {}
    for name, pipe in models.items():
        pipe.fit(X_train, y_train)
        m = evaluate_model(pipe, X_test, y_test)
        rows.append({
            "Model":     name,
            "Accuracy":  m["Accuracy"],
            "Precision": m["Precision"],
            "Recall":    m["Recall"],
            "F1":        m["F1"],
            "ROC_AUC":   m["ROC_AUC"],
        })
        fitted[name] = pipe
    results_df = (pd.DataFrame(rows)
                    .sort_values("F1", ascending=False)
                    .reset_index(drop=True))
    return results_df, fitted


# -----------------------------------------------------------------------------
# 4. A4 - Full pipeline helper
# -----------------------------------------------------------------------------
def build_full_pipeline(preprocessor, classifier):
    """Assemble preprocessing + classifier as one sklearn Pipeline."""
    return Pipeline([("preprocessing", preprocessor), ("classifier", classifier)])


# -----------------------------------------------------------------------------
# 5. A5 - LIME explanation of the pipeline
# -----------------------------------------------------------------------------
def _get_feature_names(prep):
    """Readable column names after preprocessing."""
    names = []
    for name, transformer, cols in prep.transformers_:
        if name == "num":
            names.extend(list(cols))
        elif name == "cat":
            ohe = transformer.named_steps["onehot"]
            names.extend(ohe.get_feature_names_out(cols).tolist())
    return names


def explain_with_lime(pipe, X_train, X_test, class_names,
                      instance_index=0, num_features=8, seed=42):
    """Explain one prediction using LIME on the transformed feature space."""
    prep = pipe.named_steps["preprocessing"]
    clf  = pipe.named_steps["classifier"]

    X_train_t = prep.transform(X_train)
    X_test_t  = prep.transform(X_test)
    if hasattr(X_train_t, "toarray"):
        X_train_t = X_train_t.toarray()
        X_test_t  = X_test_t.toarray()

    feature_names = _get_feature_names(prep)

    explainer = lt.LimeTabularExplainer(
        training_data=X_train_t,
        feature_names=feature_names,
        class_names=class_names,
        mode="classification",
        discretize_continuous=True,
        random_state=seed,
    )
    instance = X_test_t[instance_index]
    explanation = explainer.explain_instance(
        data_row=instance,
        predict_fn=clf.predict_proba,
        num_features=num_features,
    )
    return explanation, feature_names, X_test.iloc[[instance_index]]


# -----------------------------------------------------------------------------
# 6. O1 - SHAP feature importance (ALWAYS returns a shap.Explanation object)
# -----------------------------------------------------------------------------
def explain_with_shap(pipe, X_background, X_explain, max_samples=200):
    """Compute SHAP values, returning a proper shap.Explanation object."""
    prep = pipe.named_steps["preprocessing"]
    clf  = pipe.named_steps["classifier"]

    X_bg_t = prep.transform(X_background)
    X_ex_t = prep.transform(X_explain)
    if hasattr(X_bg_t, "toarray"):
        X_bg_t = X_bg_t.toarray()
        X_ex_t = X_ex_t.toarray()

    feature_names = _get_feature_names(prep)
    X_bg_df = pd.DataFrame(X_bg_t, columns=feature_names)
    X_ex_df = pd.DataFrame(X_ex_t, columns=feature_names)

    # Prefer the automatic path (returns Explanation for tree models)
    try:
        explainer = shap.Explainer(clf, X_bg_df, max_samples=max_samples)
        result    = explainer(X_ex_df)
        if isinstance(result, shap.Explanation):
            return result, feature_names
        # Some versions wrap in a list for multiclass; unwrap to Explanation
        if isinstance(result, list) and isinstance(result[0], shap.Explanation):
            return result[0], feature_names
    except Exception:
        pass

    # Fallback: KernelExplainer -> convert to shap.Explanation
    bg_small = X_bg_df.sample(min(50, len(X_bg_df)), random_state=42)
    ex_small = X_ex_df.sample(min(30, len(X_ex_df)), random_state=42)
    ke = shap.KernelExplainer(clf.predict_proba, bg_small)

    sv = ke.shap_values(ex_small, nsamples=100)
    if isinstance(sv, list):
        sv = sv[1]
    sv = np.asarray(sv)
    if sv.ndim == 3:
        sv = sv[:, :, 1]

    base = ke.expected_value
    if isinstance(base, (list, np.ndarray)):
        base = base[1] if len(base) > 1 else base[0]
    base = float(base)

    explanation = shap.Explanation(
        values        = sv,
        base_values   = base,
        data          = ex_small.values,
        feature_names = feature_names,
    )
    return explanation, feature_names


# -----------------------------------------------------------------------------
# 7. O3 - GridSearchCV vs RandomizedSearchCV (larger grid => real speedup)
# -----------------------------------------------------------------------------
def compare_grid_vs_random(preprocessor, X_train, y_train, cv=3, seed=42):
    """GridSearchCV vs RandomizedSearchCV on the SAME (larger) MLP space."""
    from time import perf_counter

    # 5 * 4 * 2 * 3 = 120 combinations -> Grid performs 360 fits
    search_space = {
        "clf__hidden_layer_sizes":   [(32,), (64,), (64, 32), (32, 16), (128, 64)],
        "clf__alpha":                [1e-5, 1e-4, 1e-3, 1e-2],
        "clf__activation":           ["relu", "tanh"],
        "clf__learning_rate_init":   [1e-4, 1e-3, 1e-2],
    }

    # --- Grid ---------------------------------------------------------------
    pipe_g = Pipeline([("prep", preprocessor),
                       ("clf", MLPClassifier(max_iter=300, random_state=seed))])
    gs = GridSearchCV(pipe_g, search_space, cv=cv, scoring="f1",
                      n_jobs=-1, error_score="raise")
    t0 = perf_counter(); gs.fit(X_train, y_train)
    grid_time = perf_counter() - t0

    # --- Random -------------------------------------------------------------
    pipe_r = Pipeline([("prep", preprocessor),
                       ("clf", MLPClassifier(max_iter=300, random_state=seed))])
    rs = RandomizedSearchCV(pipe_r, search_space, n_iter=15, cv=cv,
                            scoring="f1", random_state=seed, n_jobs=-1,
                            error_score="raise")
    t0 = perf_counter(); rs.fit(X_train, y_train)
    rand_time = perf_counter() - t0

    return pd.DataFrame([
        {"Search": "GridSearchCV",        "BestF1": gs.best_score_,
         "Time_s": grid_time,             "BestParams": gs.best_params_},
        {"Search": "RandomizedSearchCV",  "BestF1": rs.best_score_,
         "Time_s": rand_time,             "BestParams": rs.best_params_},
    ])


# =============================================================================
# MAIN PROGRAM
# =============================================================================
if __name__ == "__main__":
    DATA_PATH = r"D:\C drive files\Downloads\Lab Session Data (1) (1).xlsx"
    SHEET     = "marketing_campaign"

    # ---- A1: Load, clean, split
    df = load_data(DATA_PATH, SHEET)
    X, y = split_features_target(df, target="Response")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    preprocessor, num_cols, cat_cols = build_preprocessor(X_train)

    print("=== A1: Dataset Overview ===")
    print(f"Rows: {len(df)}, Features: {X.shape[1]}, Positives: {int(y.sum())}")
    print(f"Numeric cols ({len(num_cols)}): {num_cols}")
    print(f"Categorical cols ({len(cat_cols)}): {cat_cols}\n")

    # ---- A2: Hyperparameter tuning
    print("=== A2: RandomizedSearchCV for Perceptron & MLP ===")
    perc_rs = tune_perceptron(preprocessor, X_train, y_train, n_iter=15)
    mlp_rs  = tune_mlp(preprocessor, X_train, y_train, n_iter=12)
    print(f"Perceptron best F1: {perc_rs.best_score_:.4f}")
    print(f"  best params: {perc_rs.best_params_}")
    print(f"MLP best F1       : {mlp_rs.best_score_:.4f}")
    print(f"  best params: {mlp_rs.best_params_}\n")

    # ---- A3: Classifier comparison
    print("=== A3: Classifier Comparison on Hold-out Test Set ===")
    models = build_classifier_pipelines(preprocessor)
    results_df, fitted = compare_classifiers(models, X_train, y_train, X_test, y_test)
    print(results_df.to_string(index=False))
    print()

    print("--- Tuned Perceptron / MLP on the same test split ---")
    for name, rs in [("Tuned Perceptron", perc_rs), ("Tuned MLP", mlp_rs)]:
        m = evaluate_model(rs.best_estimator_, X_test, y_test)
        print(f"{name}: F1={m['F1']:.4f}, Acc={m['Accuracy']:.4f}, "
              f"ROC-AUC={m['ROC_AUC']:.4f}")
    print()

    # ---- A4: Pipeline end-to-end (RandomForest)
    print("=== A4: End-to-End Pipeline (RandomForest) ===")
    pipe_demo = build_full_pipeline(
        preprocessor,
        RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
    pipe_demo.fit(X_train, y_train)
    m_demo = evaluate_model(pipe_demo, X_test, y_test)
    print(f"Pipeline (RF) test metrics: Acc={m_demo['Accuracy']:.4f}, "
          f"F1={m_demo['F1']:.4f}, ROC-AUC={m_demo['ROC_AUC']:.4f}")
    print("Confusion matrix:\n", m_demo["Confusion"], "\n")

    # ---- A5: LIME
    if HAS_LIME:
        print("=== A5: LIME explanation (sample test instance) ===")
        expl, feat_names, inst = explain_with_lime(
            pipe_demo, X_train, X_test,
            class_names=["No Response", "Response"],
            instance_index=0, num_features=8,
        )
        print("Instance (original features):")
        print(inst.to_string(index=False))
        print("\nTop LIME features (feature -> weight):")
        for feat, w in expl.as_list():
            print(f"  {feat}: {w:+.4f}")
        expl.save_to_file("lime_explanation.html")

        # LIME figure for the report
        fig = expl.as_pyplot_figure()
        fig.set_size_inches(10, 5)
        plt.title("LIME local explanation - test instance #0")
        plt.tight_layout()
        plt.savefig("lime_explanation.png", dpi=150)
        plt.close(fig)
        print("LIME explanation saved: 'lime_explanation.html' and '.png'\n")
    else:
        print("(LIME not installed; `pip install lime`)\n")

    # ---- O1: SHAP
    if HAS_SHAP:
        print("=== O1: SHAP feature importance (RandomForest pipeline) ===")
        shap_vals, feat_names = explain_with_shap(
            pipe_demo,
            X_train.sample(200, random_state=42),
            X_test.head(50),
            max_samples=200,
        )
        try:
            vals = shap_vals.values
            if vals.ndim == 3:                       # (n, features, classes)
                vals = vals[:, :, 1]
            mean_abs = np.abs(vals).mean(axis=0)
            imp = (pd.DataFrame({"Feature": feat_names, "MeanAbsSHAP": mean_abs})
                     .sort_values("MeanAbsSHAP", ascending=False)
                     .head(15))
            print(imp.to_string(index=False))

            # Beeswarm (works now — shap_vals is a real Explanation)
            shap.plots.beeswarm(shap_vals, max_display=15, show=False)
            plt.title("SHAP summary - RandomForest pipeline")
            plt.tight_layout()
            plt.savefig("shap_summary.png", dpi=150, bbox_inches="tight")
            plt.close()
            print("SHAP summary figure saved: 'shap_summary.png'")
        except Exception as e:
            print("SHAP summary extraction failed:", e)
        print()
    else:
        print("(SHAP not installed; `pip install shap`)\n")

    # ---- O3: Grid vs Random Search (larger grid => real speedup)
    print("=== O3: GridSearchCV vs RandomizedSearchCV (MLP) ===")
    o3_df = compare_grid_vs_random(preprocessor, X_train, y_train, cv=3)
    print(o3_df[["Search", "BestF1", "Time_s"]].to_string(index=False))
    speedup = (o3_df.loc[0, "Time_s"] / o3_df.loc[1, "Time_s"]
               if o3_df.loc[1, "Time_s"] > 0 else float("nan"))
    print(f"\nRandomizedSearchCV speedup over GridSearchCV: ~{speedup:.1f}x")
    print("\n--- Best params ---")
    for _, row in o3_df.iterrows():
        print(f"{row['Search']}: {row['BestParams']}")

=== A1: Dataset Overview ===
Rows: 2240, Features: 28, Positives: 334
Numeric cols (26): ['Year_Birth', 'Income', 'Kidhome', 'Teenhome', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Cust_Year', 'Cust_Month', 'Cust_Day', 'Cust_Date_Missing']
Categorical cols (2): ['Education', 'Marital_Status']

=== A2: RandomizedSearchCV for Perceptron & MLP ===
Perceptron best F1: 0.4902
  best params: {'clf__alpha': np.float64(0.0006672367170464204), 'clf__eta0': np.float64(0.13826232179369857), 'clf__max_iter': 762, 'clf__penalty': 'elasticnet', 'clf__tol': np.float64(0.0003489018845491386)}
MLP best F1       : 0.5733
  best params: {'clf__activation': 'relu', 'clf__alpha': np.float64(1.8205657658407255e-05), 'clf__hidden_layer_sizes': (64, 

C:\Python314\Lib\site-packages\lime\explanation.py:251: ResourceWarning: unclosed file <_io.TextIOWrapper name='C:\\Python314\\Lib\\site-packages\\lime\\bundle.js' mode='r' encoding='utf8'>
  encoding="utf8").read()
Background dataset has 200 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=200 when initializing the masker.


LIME explanation saved: 'lime_explanation.html' and '.png'

=== O1: SHAP feature importance (RandomForest pipeline) ===


100%|██████████| 30/30 [00:04<00:00,  6.94it/s]


            Feature  MeanAbsSHAP
          Cust_Year     0.022645
            Recency     0.022436
       AcceptedCmp3     0.016249
    MntMeatProducts     0.010674
NumCatalogPurchases     0.009603
       AcceptedCmp5     0.009577
  NumWebVisitsMonth     0.008598
       MntGoldProds     0.008158
             Income     0.006910
          MntFruits     0.005028
  NumStorePurchases     0.004854
    MntFishProducts     0.004829
           MntWines     0.004627
       AcceptedCmp1     0.003996
      Education_PhD     0.003854
SHAP summary figure saved: 'shap_summary.png'

=== O3: GridSearchCV vs RandomizedSearchCV (MLP) ===
            Search   BestF1     Time_s
      GridSearchCV 0.577889 214.803806
RandomizedSearchCV 0.559237  24.126076

RandomizedSearchCV speedup over GridSearchCV: ~8.9x

--- Best params ---
GridSearchCV: {'clf__activation': 'relu', 'clf__alpha': 0.01, 'clf__hidden_layer_sizes': (128, 64), 'clf__learning_rate_init': 0.01}
RandomizedSearchCV: {'clf__learning_rate_init': 